# Prompting Engineering

In [13]:
# 환경설정
import os
from dotenv import load_dotenv

load_dotenv() # 현재 경로의 .env 파일을 읽어 시스템 환경변수로 등록

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

### Chat Completion

In [14]:
from openai import OpenAI

# client-server
client = OpenAI(api_key=OPENAI_API_KEY)

response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {
            "role":"system",
            "content":[
                {
                    "type":"text",
                    "text":"당신은 친절한 챗봇입니다."
                }
            ]
        },
        {
            "role":"user",
            "content":[
                {
                    "type":"text",
                    "text":"추운날 점심에는 뭘 먹으면 좋을까요?"
                }
            ]
        }
    ],
    response_format={ "type": "text" },
    temperature=1.3,    # 대답 창의성 (기본값 1) 0~2
    max_tokens=2048,    # 응답 최대 토큰수
    top_p=1             # 사용할 상위누적확률
)

print(response.choices[0].message.content)

추운 날씨에는 다음과 같은 따뜻한 음식들이 좋을 것 같습니다:

1. **된장찌개**: 구수한 된장으로 끓인 찌개는 몸을 따뜻하게 해주고, 영양가도 높습니다.
2. **김치찌개**: 매콤하고 뜨거운 김치찌개는 밥이나 면과 함께 즐기기에 좋습니다.
3. **떡국**: 부드러운 떡과 다양한 재료가 들어가 산뜻하면서도 포만감을 줍니다.
4. **칼국수**: 속을 따뜻하게 데워주는 국물과 두툼하고 쫄깃한 면발이 좋은 조화를 이룹니다.
5. **곰탕**이나 **설렁탕**: 오랜 시간 고아 낸 국물은 깊은 맛을 내며 몸을 데워줍니다.
6. **라면**: 다양한 토핑을 추가하여 간단하면서도 든든하게 먹을 수 있습니다.
7. **전골**: 다양한 재료와 함께 끓여서 많은 사람들이 함께 나누어 먹기에 좋습니다.

영양가 있는 달달한 디저트로 귤이나 수정과를 함께 곁들여 드셔보시는 것도 좋겠네요.


### 패턴1: 페르소나 & Few-Shot(기사 제목 교정)

In [15]:
from openai import OpenAI

# 함수 정의
def correct_title(query, temperature=0.3):

    # client-server
    client = OpenAI(api_key=OPENAI_API_KEY)

    # 페르소나 정의
    system_instruction="""
    당신은 신문사의 베테랑 편집장입니다. 기사들이 송고한 제목을 교정해주세요.

    ### 지시사항 ###
    - 기사의 제목이 명확하고 주제와 잘 맞도록 수정해주세요.
    - 비속어, 은어등은 제거하고 의미가 유지되도록 제목을 교정해주세요.
    - 간결하고 임팩트 있는 표현을 사용해주세요.

    ### 출력 예시 ###
    - 원래 제목 : [기사의 원래 제목]
    - 교정 제목 :
        [교정된 제목1]
        [교정된 제목2]

    ### 예시 ###
    - 원래 제목 : "어제 서울에서 큰불이 나서 수백명이 대피했다"
    - 교정 제목 :
        "서울 대형화재, 수백명 대피"
        "수백명이 큰불로 도망!"
    """

    user_message=f"""
    다음 제목을 교정해주세요.
    제목 : {query}
    """

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {
                "role":"system",
                "content":[
                    {
                        "type":"text",
                        "text":system_instruction
                    }
                ]
            },
            {
                "role":"user",
                "content":[
                    {
                        "type":"text",
                        "text":user_message
                    }
                ]
            }
        ],
        response_format={ "type": "text" },
        temperature=temperature,    # 대답 창의성 (기본값 1) 0~2
        max_tokens=2048,    # 응답 최대 토큰수
        top_p=1             # 사용할 상위누적확률
    )
# API가 여러개의 응답후보를 반환하는 경우가 있기에 response.choices[0]를 사용하여 가장 확률이 높은 첫 번째 응답을 받도록 한다.
# 받아온 응답의 message.content만 return 한다.
    return response.choices[0].message.content

print(correct_title("이은지 FM 개꿀 라디오 방송에 주목해주세요"))
print(correct_title("졸라 빡쎈 작업으로 끼니를 거르는 이 시대 노동자의 애환"))


    - 원래 제목 : 이은지 FM 개꿀 라디오 방송에 주목해주세요
    - 교정 제목 :
        "이은지 FM 라디오 방송, 주목받는 이유"
        "이은지 FM 라디오, 청취자들의 관심 집중"

    - 원래 제목 : 졸라 빡쎈 작업으로 끼니를 거르는 이 시대 노동자의 애환
    - 교정 제목 :
        "고된 작업에 끼니 거르는 현대 노동자의 고충"
        "현대 노동자, 과중한 업무로 끼니 거른다"


### 구조화된 출력 (Structured Output) - 영단어장 생성

In [ ]:
from openai import OpenAI
import json # API로부터 받은 JSON형식의 텍스트 응답을 파이썬 객체로 변환하는데 사용

# 함수 정의
def extract_eng_words(query, temperature=0.3):

    # client-server
    client = OpenAI(api_key=OPENAI_API_KEY)

    # 페르소나 정의
    system_instruction="""
    당신은 영어팝송을 이용해 흥미롭고 이해하기 쉬운 방식으로 영어를 가르치는 선생님입니다.

    # 처리단계
    1. 주어진 가사에서 자주 사용되는 영어단어 5개를 랜덤으로 추출해주세요.
    2. 각 단어의 의미를 한글로 알려주세요.
    3. 단어별로 유사한 단어도 함께 소개해주세요.

    # 출력형식
    출력형식은 json 형식입니다.
    - 최상위 키는 "json_list"여야 합니다.
    - "json_list"의 값은 단어별 json 객체들이 담긴 리스트여야 합니다.

    # 출력형식예시
    {
        "json_list" :[
            {
                "단어" : "yesterday",
                "의미" : "어제"
                "유사어" : [
                    {
                        "유사단어" : "today"
                    },
                    {
                        "유사단어" : "tommorow"
                    }
                ]
            }
        ]
    }
    """

    user_message=f"""
    노래가사: {query}
    """

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {
                "role":"system",
                "content":[
                    {
                        "type":"text",
                        "text":system_instruction
                    }
                ]
            },
            {
                "role":"user",
                "content":[
                    {
                        "type":"text",
                        "text":user_message
                    }
                ]
            }
        ],
        response_format={ "type": "json_object" },
        temperature=temperature,    # 대답 창의성 (기본값 1) 0~2
        max_tokens=2048,    # 응답 최대 토큰수
        top_p=1             # 사용할 상위누적확률
    )

    return json.loads(response.choices[0].message.content) # 응답 json 문자열을 파이썬 객체로 변환

lyrics = "Yesterday all my troubles seemed so far away Now it looks as though they're here to stay Oh, I believe in yesterday Suddenly I'm not half the man I used to be There's a shadow hanging over me Oh, yesterday came suddenly Why she had to go I don't know, she wouldn't say I said something wrong now I long for yesterday Yesterday love was such an easy game to play Now I need a place to hide away Oh, I believe in yesterday. Why she had to go I don't know, She wouldn't say I said something wrong Now I long for yesterday Yesterday love was such an easy game to play Now I need a place to hide away Oh, I believe in yesterday"

result = extract_eng_words(lyrics, temperature=1)

words = result['json_list']

# 파이썬 객체로 변환되었기 때문에 for 반복문을 통해 각 요소를 순회하고, 딕셔너리의 키를 이용해 원하는 데이터에 직접 접근 가능.

for word_dict in words:
    print(f"단어 : {word_dict["단어"]}")
    print(f"의미 : {word_dict["의미"]}")



단어 : yesterday
의미 : 어제
단어 : troubles
의미 : 문제들, 걱정들
단어 : believe
의미 : 믿다
단어 : suddenly
의미 : 갑자기
단어 : shadow
의미 : 그림자


### 생각의 사슬(Chain-of-Thought) (냉털 마스터)
- Reason + Act 기법으로 현재 상황에 대한 통찰이후, 다음 행동에 대한 작성을 유도하는 기법.

In [ ]:
from openai import OpenAI

# 함수 정의
def my_refregator(query, temperature=0.3):

    # client-server
    client = OpenAI(api_key=OPENAI_API_KEY)

    # 페르소나 정의
    system_instruction="""
    너는 냉장고에 있는 재료들을 활용하여 창의적이고 실용적인 저녁식사 아이디어를 제안하는 요리 전문가이다.
    네 역할은 사용자로부터 제공받은 재료목록을 분석하고, 이를 활용할 수 있는 요리 아이디어를 구상해서,
    조리방법을 단계별로 상세히 설명하는 것이다.

    # 출력예시
    1. 상황분석
    - 현재 가진 재료는 [사용자가 입력한 재료] 입니다.
    - 주재료인 닭고기와 다양한 채소들이 있습니다.
    - 이 재료들은 스튜, 볶음, 찜 요리에 적합합니다.

    2. 헹동계획
    - 가장 쉽게 만들 수 있는 요리로 닭볶음탕을 제안합니다.
    - 각 요리에 필요한 재료와 조리도구를 확인합니다.
    - 부족한 재료가 있다면, 대체 가능한 옵션을 제시합니다.
    - 조리 과정을 단계별로 상세히 설명합니다.
    - 맛을 향상시킬 수 있는 팁과 주의사항 또한 제공합니다.

    3. 실행
    - 여기에 상세레시피를 단계별로 작성합니다.
    """

    user_message=f"""
    사용자의 냉장고에는 현재 {query}가 있습니다.
    """

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {
                "role":"system",
                "content":[
                    {
                        "type":"text",
                        "text":system_instruction
                    }
                ]
            },
            {
                "role":"user",
                "content":[
                    {
                        "type":"text",
                        "text":user_message
                    }
                ]
            }
        ],
        response_format={ "type": "text" },
        temperature=temperature,    # 대답 창의성 (기본값 1) 0~2
        max_tokens=2048,    # 응답 최대 토큰수
        top_p=1             # 사용할 상위누적확률
    )

    return response.choices[0].message.content

print(my_refregator("공기(air), 노력, 열정", temperature=1))




1. 상황분석
- 현재 가진 재료는 공기, 노력, 열정입니다.
- 이 재료들은 물리적 식재료가 아니지만, 창의적인 요리 아이디어를 상징적으로 얻을 수 있습니다.
- 따라서 이를 활용해 상상 속에서 완벽한 가상의 요리를 만들어볼 수 있습니다.

2. 헹동계획
- 가장 고무적이고 흥미로운 요리로 ‘상상의 만찬’을 제안합니다.
- 필요재료: 공기와 노력, 열정 (상상력과 결단력을 동원하여 시작!)
- 단계별로 조리 과정을 통해 가상의 요리를 완성합니다.

3. 실행
1단계: 상상력의 불을 켜세요. 자신이 원하는 완벽한 저녁 식사를 머릿속에 그려봅니다. 
2단계: 노력(우리가 만들고자 하는 욕구)의 양념을 추가합니다. 상상하는 요리에 필요한 노력을 마치 현실에서의 재료처럼 투입하세요.
3단계: 열정(마음의 온도)을 가합니다. 이 열정은 가상의 오븐에서 요리를 완벽히 익히는 열입니다.
4단계: 이제 모든 과정을 떠올리며 만족한 미소로 상상의 만찬을 즐겨보세요. 

- 팁: 현실에서 요리를 만들고 싶다면, 냉장고를 열고 실제 재료를 구입하여 창의적인 요리를 시작해보세요! 현실 속의 열정과 노력을 더하면 무슨 요리든 가능합니다.


### 면접 질문 생성

In [30]:
from openai import OpenAI

# 함수 정의
def job_inteview(job_posting, temperature=0.3):

    # client-server
    client = OpenAI(api_key=OPENAI_API_KEY)

    # 페르소나 정의
    system_instruction="""
    당신은 풀스택과 AI 개발 분야의 전문가이면서, 해당 분야의 면접 전문가입니다.
    사용자가 제공한 구인공고에 근거하여 핵심적인 질문들로 지원자를 평가할 수 있어야 합니다.
    """

    user_message=f"""
    아래 채용공고의 직무에 대해 예상면접 질문과 모법답안을 작성해주세요.

    -- 가이드 --
    하드스킬과 소프트스킬2개의 섹션으로 나눠 작성해주세요.
    각 섹션별로 2개의 질문과 답변을 준비해주세요.

    -- 출력형식 --
    # 1. 하드스킬
    질문1)
    답변1)
    
    질문2)
    답변2

    # 2. 소프트스킬
    질문1)
    답변1)

    질문2)
    답변2)

    -- 채용공고 --
    {job_posting}
    """

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {
                "role":"system",
                "content":[
                    {
                        "type":"text",
                        "text":system_instruction
                    }
                ]
            },
            {
                "role":"user",
                "content":[
                    {
                        "type":"text",
                        "text":user_message
                    }
                ]
            }
        ],
        response_format={ "type": "text" },
        temperature=temperature,    # 대답 창의성 (기본값 1) 0~2
        max_tokens=2048,    # 응답 최대 토큰수
        top_p=1             # 사용할 상위누적확률
    )

    return response.choices[0].message.content

job_posting = """
부서소개
- 최고의 IT 기술력과 노하우를 바탕으로 끊임없이 연구 개발하여 고객이 만족할 수 있는 최적의 금융 솔루션을 제공하는 Professional Software Development Team!

핵심가치
- Innovation + Cooperation + Ownership + Concentration

부서의 미션
- 서비스 경쟁력 강화 및 시장 지배력 강화
- 프로젝트 관리 효율성 제고
- 데이터 통합 및 업무 프로세스 개선
- 내 외부 사용자 만족을 위한 최적의 솔루션 제공
- 급변하는 IT 트렌드에 대응하는 기술 기반 마련

부서에서 하는 일
- Client & Server 소프트웨어 개발 및 운영
- 데이터 수집, 가공 자동화 및 관리 시스템 개발
- IT 기술과 금융업무의 전문성을 바탕으로 대외 프로젝트 진행
- 차세대 기술 기반 확보 및 상품 적용 방안 수립
- 빅데이터 & 텍스트 마이닝을 통한 정보의 고급화 개발
- REST-ful API, TCP/IP 통신 데이터 서비스 개발

부서의 자랑거리, 우리 부서만의 문화
- Client/Server, 내/외부 프로그램 등 다양한 분야를 아우르는 시스템 개발 업무 수행
- 믿음과 단합으로 이루어진 수평적 조직 문화
- 자유로운 의견 개진과 공유를 통한 창의적 개발 활동

추가사항
- 여러 개발 언어를 활용하지만 C#을 주로 사용합니다.
- 응용 프로그램, 웹 등 다양한 분야에 개발에 관심을 가지고, 새로운 영역에 도전하고 학습하는 자세를 지향합니다.
담당업무
- 소프트웨어 개발 및 운영


자격요건
- 학사 이상

- 경력 : 3년 이하

- 컴퓨터 관련 전공 또는 부트캠프 등 실무 중심 개발 교육 이수하신 분

- .NET, MS-SQL 프로그래밍 능력 보유하신 분


우대사항
- 데이터 조회 성능 향상에 대한 관심과 기초 튜닝 경험 보유하신 분

- REST API, FTP/sFTP, TCP/IP, gRPC 등 다양한 통신 방식 개발 경험 보유하신 분  

- LLM API, 챗봇 개발 등 신기술 활용에 열려 있으신 분


공통 우대사항
- 주식시장과 기업공시에 대한 기본 지식을 갖추신분

- 빠른 면접진행 및 출근가능자 우대


기타사항
- 긍정 마인드, 책임감, 솔선수범
"""

print(job_inteview(job_posting, temperature=1))

# 1. 하드스킬

질문1)  
.NET과 MS-SQL을 사용하여 개발한 프로젝트 중 가장 도전적이었던 경험에 대해 말씀해 주세요.

답변1)  
제가 진행한 프로젝트 중 가장 도전적이었던 경험은, 기존의 레거시 시스템을 .NET과 MS-SQL로 전환하는 작업이었습니다. 처음에는 데이터베이스 구조를 새롭게 설계해야 했고, 이를 효율적으로 마이그레이션하는 것이 매우 도전적이었습니다. 성능 최적화를 위해 인덱싱을 개선하고, 쿼리를 최적화하여 데이터 조회 속도를 30% 이상 향상시켰습니다. 이를 통해 사용자의 응답 속도를 크게 개선할 수 있었습니다.

질문2)  
REST API, FTP/sFTP, TCP/IP, gRPC 등의 통신 방식 개발 경험이 있으신데, 그 중 가장 많이 사용한 것과 그 이유를 설명해 주세요.

답변2)  
개발 프로젝트 중 REST API를 가장 많이 사용했습니다. REST API는 웹 서비스에 최적화된 형식으로, HTTP를 기반으로 하여 다른 시스템과의 통신을 원활하게 해줍니다. 다양한 클라이언트와 쉽게 통신이 가능하고, 표준화된 규격을 가지고 있어 API 설계 및 관리가 용이한 점이 주요 장점이었습니다. 특히, 금융 데이터의 안전한 전송을 위해 관련 인증 및 보안 레이어를 추가하여 사용했습니다.

# 2. 소프트스킬

질문1)  
팀에서 주로 C#을 사용하는데, 새로운 개발 언어를 학습한 경험과 그에 따른 도전 과제는 무엇이었는지 말씀해 주세요.

답변1)  
제가 가장 최근에 학습한 언어는 Python이었습니다. 데이터 분석과 머신러닝 프로젝트에 참여하면서 학습 필요성을 느꼈습니다. 처음에는 문법 차이와 새로운 라이브러리 사용법 때문에 어려움이 있었지만, 온라인 강의와 프로젝트 예제를 통해 빠르게 개선했습니다. 새로운 언어를 배우는 과정을 통해 기술적으로 유연해질 수 있었고, 이를 통해 팀에 새로운 아이디어를 제안할 수 있었습니다.

질문2)  
수평적인 조직 문화에서 자유롭게 의견을 제시하고 공유하는 것이 중요한데, 실제로 팀 내에서